# 🚀 Mission Artemis 2 - Telemetry Simulator

This notebook simulates the complete Artemis 2 mission telemetry, synchronized with a 4-minute (240 second) mission timeline.

**Data Generated:**
- Vehicle Telemetry (altitude, velocity, acceleration, fuel)
- Crew Vitals (heart rate, blood pressure, O2 saturation for 4 astronauts)
- Environmental Conditions (cabin pressure, temperature, radiation)
- Mission Events (phase changes, milestones, alerts)

In [ ]:
# Configuration - Update these values
EVENTSTREAM_ENDPOINT = "<YOUR_CUSTOM_ENDPOINT_URL>"
MISSION_DURATION_SECONDS = 240  # 4-minute mission
TELEMETRY_INTERVAL_MS = 500     # Send data every 500ms

In [ ]:
import json
import uuid
import time
import random
import math
from datetime import datetime
from typing import Dict, List
import requests

In [ ]:
# Mission phases with timing (seconds)
MISSION_PHASES = [
    {"name": "Pre-Launch", "start": 0, "end": 30},
    {"name": "Launch", "start": 30, "end": 60},
    {"name": "TLI", "start": 60, "end": 90},
    {"name": "Cruise", "start": 90, "end": 120},
    {"name": "LunarOrbit", "start": 120, "end": 180},
    {"name": "TEI", "start": 180, "end": 210},
    {"name": "Re-entry", "start": 210, "end": 230},
    {"name": "Splashdown", "start": 230, "end": 240}
]

# Crew roster
CREW = [
    {"id": "CMD", "name": "Commander Reid"},
    {"id": "PLT", "name": "Pilot Hansen"},
    {"id": "MS1", "name": "Specialist Torres"},
    {"id": "MS2", "name": "Engineer Kim"}
]

# Mission milestones
MILESTONES = {
    0: ("Mission Start", "All systems nominal, countdown initiated"),
    30: ("Liftoff", "Main engines ignition, vehicle has cleared the tower"),
    45: ("Max-Q", "Maximum dynamic pressure achieved"),
    55: ("MECO", "Main Engine Cutoff, entering coast phase"),
    60: ("TLI Burn Start", "Trans-Lunar Injection burn initiated"),
    88: ("TLI Complete", "On trajectory to the Moon"),
    120: ("Lunar Orbit Insertion", "Entering lunar orbit"),
    180: ("TEI Burn", "Trans-Earth Injection, heading home"),
    210: ("Entry Interface", "Beginning atmospheric re-entry"),
    225: ("Chute Deploy", "Main parachutes deployed"),
    238: ("Splashdown", "Successful water landing!")
}

In [ ]:
class ArtemisSimulator:
    """Simulates Artemis 2 mission telemetry"""
    
    def __init__(self, endpoint: str):
        self.endpoint = endpoint
        self.session_id = str(uuid.uuid4())
        self.events_sent = 0
    
    def get_phase(self, mission_time: float) -> str:
        """Get current mission phase"""
        for phase in MISSION_PHASES:
            if phase["start"] <= mission_time < phase["end"]:
                return phase["name"]
        return "Complete"
    
    def send_event(self, event: Dict, event_type: str):
        """Send event to Eventstream"""
        event["_eventType"] = event_type  # For routing
        try:
            response = requests.post(
                self.endpoint,
                headers={"Content-Type": "application/json"},
                json=event,
                timeout=5
            )
            self.events_sent += 1
            if response.status_code >= 400:
                print(f"Error: {response.status_code}")
        except Exception as e:
            print(f"Send failed: {e}")
    
    def generate_vehicle_telemetry(self, mission_time: float) -> Dict:
        """Generate vehicle telemetry based on mission phase"""
        phase = self.get_phase(mission_time)
        
        # Altitude profile (simplified)
        if phase == "Pre-Launch":
            altitude = 0
            velocity = 0
            acceleration = 0
        elif phase == "Launch":
            t = mission_time - 30
            altitude = 0.5 * 30 * t**2 / 1000  # km
            velocity = 30 * t * 3.6  # km/h
            acceleration = 3 + random.uniform(-0.2, 0.2)
        elif phase == "TLI":
            altitude = 200 + (mission_time - 60) * 100  # km
            velocity = 28000 + (mission_time - 60) * 200  # km/h
            acceleration = 2 + random.uniform(-0.1, 0.1)
        elif phase in ["Cruise", "LunarOrbit"]:
            altitude = 200000 + (mission_time - 90) * 1000
            velocity = 35000 + random.uniform(-100, 100)
            acceleration = 0 + random.uniform(-0.01, 0.01)
        elif phase == "TEI":
            altitude = 384400 - (mission_time - 180) * 5000
            velocity = 35000 + (mission_time - 180) * 100
            acceleration = 1.5
        elif phase == "Re-entry":
            altitude = max(0, 150000 - (mission_time - 210) * 7500)
            velocity = max(0, 40000 - (mission_time - 210) * 2000)
            acceleration = 4 + random.uniform(-0.5, 0.5)
        else:  # Splashdown
            altitude = max(0, 10 - (mission_time - 230))
            velocity = max(0, 50 - (mission_time - 230) * 5)
            acceleration = 0.5
        
        return {
            "EventId": str(uuid.uuid4()),
            "Timestamp": datetime.utcnow().isoformat() + "Z",
            "MissionTime": mission_time,
            "Phase": phase,
            "Altitude": round(altitude, 2),
            "Velocity": round(velocity, 2),
            "Acceleration": round(acceleration, 3),
            "FuelRemaining": round(100 - mission_time * 0.3 + random.uniform(-1, 1), 1),
            "HullTemperature": round(20 + (acceleration * 50) + random.uniform(-5, 5), 1),
            "Heading": round(random.uniform(0, 360), 1),
            "DistanceFromEarth": round(altitude, 2),
            "DistanceFromMoon": round(max(0, 384400 - altitude), 2),
            "SessionId": self.session_id
        }
    
    def generate_crew_vitals(self, mission_time: float, crew: Dict) -> Dict:
        """Generate crew vital signs"""
        phase = self.get_phase(mission_time)
        
        # Base heart rate varies by phase
        if phase in ["Launch", "Re-entry"]:
            base_hr = 95
            stress = "Elevated"
        elif phase in ["TLI", "TEI"]:
            base_hr = 85
            stress = "Normal"
        else:
            base_hr = 70
            stress = "Low"
        
        return {
            "EventId": str(uuid.uuid4()),
            "Timestamp": datetime.utcnow().isoformat() + "Z",
            "MissionTime": mission_time,
            "CrewId": crew["id"],
            "CrewName": crew["name"],
            "HeartRate": base_hr + random.randint(-10, 15),
            "BloodPressureSystolic": 120 + random.randint(-10, 20),
            "BloodPressureDiastolic": 80 + random.randint(-5, 10),
            "OxygenSaturation": round(98 + random.uniform(-2, 1), 1),
            "BodyTemperature": round(36.6 + random.uniform(-0.3, 0.5), 1),
            "RespirationRate": 14 + random.randint(-2, 4),
            "StressLevel": stress,
            "SessionId": self.session_id
        }
    
    def generate_environmental(self, mission_time: float) -> Dict:
        """Generate environmental conditions"""
        phase = self.get_phase(mission_time)
        
        # Radiation increases during cruise/lunar phases
        if phase in ["Cruise", "LunarOrbit"]:
            radiation = 0.5 + random.uniform(0, 0.3)
        else:
            radiation = 0.1 + random.uniform(0, 0.1)
        
        return {
            "EventId": str(uuid.uuid4()),
            "Timestamp": datetime.utcnow().isoformat() + "Z",
            "MissionTime": mission_time,
            "CabinPressure": round(101.3 + random.uniform(-0.5, 0.5), 2),
            "CabinTemperature": round(22 + random.uniform(-1, 1), 1),
            "CabinHumidity": round(45 + random.uniform(-5, 5), 1),
            "CO2Level": round(400 + random.uniform(-20, 50), 1),
            "O2Level": round(21 + random.uniform(-0.5, 0.5), 2),
            "RadiationLevel": round(radiation, 3),
            "NoiseLevel": round(60 + random.uniform(-5, 15), 1),
            "Location": "CrewCompartment",
            "SessionId": self.session_id
        }
    
    def generate_mission_event(self, mission_time: float, name: str, desc: str) -> Dict:
        """Generate mission event"""
        return {
            "EventId": str(uuid.uuid4()),
            "Timestamp": datetime.utcnow().isoformat() + "Z",
            "MissionTime": mission_time,
            "EventType": "Milestone",
            "EventName": name,
            "Description": desc,
            "Severity": "Info",
            "Phase": self.get_phase(mission_time),
            "Source": "FlightControl",
            "SessionId": self.session_id
        }
    
    def run_mission(self):
        """Run complete mission simulation"""
        print(f"🚀 Starting Artemis 2 Mission - Session: {self.session_id[:8]}...")
        print(f"   Duration: {MISSION_DURATION_SECONDS} seconds")
        print()
        
        start_time = time.time()
        last_phase = None
        
        while True:
            elapsed = time.time() - start_time
            mission_time = elapsed * (MISSION_DURATION_SECONDS / MISSION_DURATION_SECONDS)  # Real-time
            
            if mission_time >= MISSION_DURATION_SECONDS:
                break
            
            # Check for phase change
            current_phase = self.get_phase(mission_time)
            if current_phase != last_phase:
                print(f"  🔄 Phase: {current_phase} (T+{int(mission_time)}s)")
                last_phase = current_phase
            
            # Check for milestones
            mt_int = int(mission_time)
            if mt_int in MILESTONES:
                name, desc = MILESTONES[mt_int]
                print(f"  📍 {name}: {desc}")
                event = self.generate_mission_event(mission_time, name, desc)
                self.send_event(event, "MissionEvents")
                MILESTONES.pop(mt_int)  # Only send once
            
            # Generate and send telemetry
            vehicle = self.generate_vehicle_telemetry(mission_time)
            self.send_event(vehicle, "VehicleTelemetry")
            
            # Crew vitals (all 4 astronauts)
            for crew in CREW:
                vitals = self.generate_crew_vitals(mission_time, crew)
                self.send_event(vitals, "CrewVitals")
            
            # Environmental (every other tick)
            if self.events_sent % 2 == 0:
                env = self.generate_environmental(mission_time)
                self.send_event(env, "EnvironmentalConditions")
            
            time.sleep(TELEMETRY_INTERVAL_MS / 1000)
        
        print()
        print(f"✅ Mission complete! {self.events_sent} events sent.")
        print(f"   Session ID: {self.session_id}")

In [ ]:
# Run the mission!
if EVENTSTREAM_ENDPOINT.startswith("<"):
    print("⚠️ Please update EVENTSTREAM_ENDPOINT with your Custom Endpoint URL")
else:
    simulator = ArtemisSimulator(EVENTSTREAM_ENDPOINT)
    simulator.run_mission()

## 📊 Verify Data in KQL

Run these queries to analyze mission telemetry:

```kql
// Vehicle telemetry by phase
VehicleTelemetry
| summarize AvgAltitude=avg(Altitude), MaxVelocity=max(Velocity), MaxG=max(Acceleration) by Phase
| order by AvgAltitude asc

// Crew heart rates during launch
CrewVitals
| where MissionTime between (30 .. 60)
| summarize AvgHR=avg(HeartRate), MaxHR=max(HeartRate) by CrewName
| render columnchart

// Radiation exposure during mission
EnvironmentalConditions
| summarize AvgRadiation=avg(RadiationLevel) by bin(MissionTime, 30)
| render linechart

// Mission timeline
MissionEvents
| project MissionTime, EventName, Description
| order by MissionTime asc
```